In [ ]:
import hashlib
import math
import re
import sys
import time
import tracemalloc
import random
import matplotlib.pyplot as plt

print("Đã nạp xong các thư viện cần thiết!")

Đã nạp xong các thư viện cần thiết!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def count_trailing_zeros(n):
    """Đếm số lượng bit 0 liên tiếp ở cuối chuỗi nhị phân của n"""
    if n == 0:
        return 32  # Giới hạn 32 bit
    zeros = 0
    while (n & 1) == 0:
        zeros += 1
        n >>= 1
    return zeros

def hash_function(data_str, seed):
    """
    Tạo ra hàm băm độc lập bằng cách kết hợp MD5 với seed.
    Trả về số nguyên 32-bit không âm.
    """
    salted_input = f"{seed}_{data_str}".encode('utf-8')
    hash_digest = hashlib.md5(salted_input).hexdigest()
    return int(hash_digest[:8], 16)

print("Đã định nghĩa xong các hàm băm toán học!")

Đã định nghĩa xong các hàm băm toán học!


In [ ]:
class FlajoletMartinBasic:
    """Thuật toán Flajolet-Martin cơ bản (Dùng 1 hàm băm)"""
    def __init__(self, seed=42):
        self.seed = seed
        self.max_zeros = 0
        self.phi = 0.77351  # Hằng số hiệu chỉnh Flajolet-Martin

    def update(self, item):
        hash_val = hash_function(item, self.seed)
        r = count_trailing_zeros(hash_val)
        if r > self.max_zeros:
            self.max_zeros = r

    def estimate(self):
        return (2 ** self.max_zeros) / self.phi


class FlajoletMartinAdvanced:
    """
    Thuật toán Flajolet-Martin cải tiến:
    Sử dụng m hàm băm + Kỹ thuật Median of Means để thu hẹp sai số
    """
    def __init__(self, num_hashes=128, num_groups=16):
        self.num_hashes = num_hashes
        self.num_groups = num_groups
        self.hashes_per_group = num_hashes // num_groups
        self.max_zeros = [0] * num_hashes
        self.phi = 0.77351

    def update(self, item):
        for i in range(self.num_hashes):
            hash_val = hash_function(item, seed=i*100 + 7)
            r = count_trailing_zeros(hash_val)
            if r > self.max_zeros[i]:
                self.max_zeros[i] = r

    def estimate(self):
        group_averages = []
        for g in range(self.num_groups):
            start_idx = g * self.hashes_per_group
            end_idx = start_idx + self.hashes_per_group
            group_zeros = self.max_zeros[start_idx:end_idx]

            # Tính trung bình kết quả ước lượng trong từng nhóm (Mean)
            avg_r = sum(group_zeros) / len(group_zeros)  # Tính trung bình số bit 0 trước
            avg_est = (2 ** avg_r) / self.phi            # Rồi mới mũ hóa để ra ước lượng nhóm
            group_averages.append(avg_est)


        # Lấy trung vị của các nhóm (Median) để loại bỏ giá trị cực đoan
        group_averages.sort()
        mid = len(group_averages) // 2
        if len(group_averages) % 2 == 0:
            return (group_averages[mid - 1] + group_averages[mid]) / 2.0
        else:
            return group_averages[mid]

print("Đã cài đặt xong các lớp Flajolet-Martin!")

Đã cài đặt xong các lớp Flajolet-Martin!


In [ ]:
def stream_log_file(file_path):
    """Đọc file log thật theo dạng luồng bằng split chuỗi"""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line:
                # Lấy phần tử đầu tiên trước dấu cách (chính là địa chỉ IP)
                ip = line.split(' ', 1)[0]
                yield ip

print("Đã sẵn sàng bộ đọc luồng dữ liệu!")

Đã sẵn sàng bộ đọc luồng dữ liệu!


In [ ]:
def run_experiment(log_file_path=None, sample_step=100000):
    fm_basic = FlajoletMartinBasic(seed=42)
    fm_advanced = FlajoletMartinAdvanced(num_hashes=128, num_groups=16)
    exact_set = set()

    stream_counts = []
    exact_unique_history = []
    fm_basic_history = []
    fm_adv_history = []

    if log_file_path:
        print(f"Đang đọc dữ liệu từ file: {log_file_path}")
        stream_data = stream_log_file(log_file_path)
    else:
        print("Không có đường dẫn file log.")
        exit

    tracemalloc.start()
    start_time = time.time()

    total_processed = 0
    print("Bắt đầu xử lý luồng dữ liệu thời gian thực...\n")

    # --------------------------------------------------------------------------
    # BƯỚC 1: XỬ LÝ LUỒNG DỮ LIỆU
    # --------------------------------------------------------------------------
    for ip in stream_data:
        total_processed += 1

        exact_set.add(ip)
        fm_basic.update(ip)
        fm_advanced.update(ip)

        if total_processed % sample_step == 0:
            stream_counts.append(total_processed)
            exact_unique_history.append(len(exact_set))
            fm_basic_history.append(fm_basic.estimate())
            fm_adv_history.append(fm_advanced.estimate())

            print(f"Đã xử lý: {total_processed:,} dòng | "
                  f"Thực tế (Set): {len(exact_set):,} | "
                  f"FM (1 Hash): {int(fm_basic.estimate()):,} | "
                  f"FM Cải tiến (128 Hash): {int(fm_advanced.estimate()):,}")

    elapsed_time = time.time() - start_time
    current_ram, peak_ram = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # --------------------------------------------------------------------------
    # BƯỚC 2: SAU KHI XỬ LÝ XONG TOÀN BỘ MỚI TÍNH TOÁN KẾT LUẬN & ĐỘ CHÍNH XÁC
    # --------------------------------------------------------------------------
    exact_final = len(exact_set)
    basic_final = fm_basic.estimate()
    adv_final = fm_advanced.estimate()

    # Tính sai số và độ chính xác cuối cùng
    err_basic = abs(basic_final - exact_final) / exact_final * 100
    err_adv = abs(adv_final - exact_final) / exact_final * 100

    acc_basic = max(0.0, 100.0 - err_basic)
    acc_adv = max(0.0, 100.0 - err_adv)

    # Tính toán chính xác dung lượng RAM thực tế dựa trên số lượng exact_final vừa đếm được
    set_container_bytes = sys.getsizeof(exact_set)
    set_elements_bytes = sum(sys.getsizeof(s) for s in exact_set)
    total_set_bytes = set_container_bytes + set_elements_bytes

    fm_basic_bytes = sys.getsizeof(fm_basic) + sys.getsizeof(fm_basic.max_zeros)
    fm_adv_bytes = sys.getsizeof(fm_advanced) + sys.getsizeof(fm_advanced.max_zeros) + sum(sys.getsizeof(x) for x in fm_advanced.max_zeros)

    # In kết quả tổng kết cuối cùng với các biến động hoàn toàn
    print("\n" + "="*70)
    print("KẾT QUẢ THỰC NGHIỆM TỔNG KẾT (BÁO CÁO NHÓM 16)")
    print("="*70)
    print("1. TỔNG QUAN XỬ LÝ:")
    print(f"   - Tổng số dòng log đã quét:            {total_processed:,}")
    print(f"   - Số IP Duy nhất CHÍNH XÁC (Set):      {exact_final:,}")
    print(f"   - Thời gian thực thi:                  {elapsed_time:.2f} giây")
    print("\n2. ĐÁNH GIÁ ĐỘ CHÍNH XÁC VÀ SAI SỐ CUỐI CÙNG:")
    print(f"   - FM Cơ bản (1 Hash):                  {int(basic_final):,} | Sai số: {err_basic:.2f}% | Độ chính xác: {acc_basic:.2f}%")
    print(f"   - FM Cải tiến (128 Hash):              {int(adv_final):,} | Sai số: {err_adv:.2f}% | Độ chính xác: {acc_adv:.2f}%")
    print("\n3. PHÂN TÍCH BỘ NHỚ RAM THỰC TẾ CHIẾM DỤNG:")
    print(f"   - Peak RAM tổng thể của hệ thống:     {peak_ram / (1024*1024):.4f} MB")
    print("   ------------------------------------------------------------------")
    print("   a) Cấu trúc 'exact_set' (Set kiểm thử):")
    print(f"      + Không gian bảng băm (Set Container): {set_container_bytes / (1024*1024):.4f} MB ({set_container_bytes:,} Bytes)")
    print(f"      + Dung lượng {exact_final:,} chuỗi IP thực tế: {set_elements_bytes / (1024*1024):.4f} MB ({set_elements_bytes:,} Bytes)")
    print(f"      + TỔNG CỘNG Set chiếm:               {total_set_bytes / (1024*1024):.4f} MB ({total_set_bytes:,} Bytes)")
    print(f"      => Chiếm {(total_set_bytes / peak_ram) * 100:.2f}% TỔNG RAM HỆ THỐNG.")
    print("\n   b) Cấu trúc Flajolet-Martin:")
    print(f"      + FM Cơ bản (1 Hash):                 {fm_basic_bytes:,} Bytes ({fm_basic_bytes / 1024:.2f} KB)")
    print(f"      + FM Cải tiến (128 Hash):             {fm_adv_bytes:,} Bytes ({fm_adv_bytes / 1024:.2f} KB)")
    print(f"      => Bộ nhớ FM 128 Hash tiết kiệm hơn Set: {total_set_bytes / fm_adv_bytes:,.1f} LẦN!")
    print("="*70)

    # --------------------------------------------------------------------------
    # BƯỚC 3: VẼ 3 BIỂU ĐỒ TRỰC QUAN HÓA KẾT QUẢ CUỐI CÙNG
    # --------------------------------------------------------------------------
    plt.figure(figsize=(18, 5))

    # Biểu đồ 1: Số lượng IP duy nhất ước lượng vs Thực tế
    plt.subplot(1, 3, 1)
    plt.plot(stream_counts, exact_unique_history, 'k-', label='Chính xác (Set)', linewidth=2.5)
    plt.plot(stream_counts, fm_basic_history, 'r--', label='FM Cơ bản (1 Hash)', alpha=0.7)
    plt.plot(stream_counts, fm_adv_history, 'g-', label='FM Cải tiến (128 Hash)', linewidth=2)
    plt.title('Tăng trưởng Số lượng IP Duy nhất', fontsize=11, fontweight='bold')
    plt.xlabel('Số dòng log đã xử lý')
    plt.ylabel('Số lượng IP duy nhất')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    # Biểu đồ 2: So sánh Tỷ lệ Sai số tương đối (%)
    plt.subplot(1, 3, 2)
    errors_basic_steps = [abs(b - e)/e * 100 for b, e in zip(fm_basic_history, exact_unique_history)]
    errors_adv_steps = [abs(a - e)/e * 100 for a, e in zip(fm_adv_history, exact_unique_history)]

    plt.plot(stream_counts, errors_basic_steps, 'r-o', label='FM 1 Hash (%)')
    plt.plot(stream_counts, errors_adv_steps, 'g-s', label='FM `128 Hash (%)')
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.title('Diễn biến Sai số tương đối (%)', fontsize=11, fontweight='bold')
    plt.xlabel('Số dòng log đã xử lý')
    plt.ylabel('Sai số (%)')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    # Biểu đồ 3: So sánh Độ chính xác (Accuracy %)
    plt.subplot(1, 3, 3)
    acc_basic_steps = [max(0.0, 100.0 - err) for err in errors_basic_steps]
    acc_adv_steps = [max(0.0, 100.0 - err) for err in errors_adv_steps]

    plt.plot(stream_counts, acc_basic_steps, 'r-o', label='FM 1 Hash (%)')
    plt.plot(stream_counts, acc_adv_steps, 'g-s', label='FM 128 Hash (%)')
    plt.axhline(y=100, color='black', linestyle='--', alpha=0.5)
    plt.title('Diễn biến Độ chính xác (Accuracy %)', fontsize=11, fontweight='bold')
    plt.xlabel('Số dòng log đã xử lý')
    plt.ylabel('Độ chính xác (%)')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

print("Đã cập nhật hàm run_experiment hoàn chỉnh!")

Đã cập nhật hàm run_experiment hoàn chỉnh!


In [ ]:
# ==============================================================================
# CHẠY THỰC NGHIỆM
# ==============================================================================

# LOG_FILE = "/content/drive/MyDrive/zanbil.log"

LOG_FILE = "/content/drive/MyDrive/accessLog/access.log"

run_experiment(log_file_path=LOG_FILE, sample_step=50000)

Đang đọc dữ liệu từ file: /content/drive/MyDrive/accessLog/access.log
Bắt đầu xử lý luồng dữ liệu thời gian thực...

Đã xử lý: 50,000 dòng | Thực tế (Set): 2,179 | FM (1 Hash): 2,647 | FM Cải tiến (128 Hash): 4,098
Đã xử lý: 100,000 dòng | Thực tế (Set): 3,987 | FM (1 Hash): 2,647 | FM Cải tiến (128 Hash): 6,297
Đã xử lý: 150,000 dòng | Thực tế (Set): 5,346 | FM (1 Hash): 2,647 | FM Cải tiến (128 Hash): 8,939
Đã xử lý: 200,000 dòng | Thực tế (Set): 6,510 | FM (1 Hash): 2,647 | FM Cải tiến (128 Hash): 9,711
Đã xử lý: 250,000 dòng | Thực tế (Set): 7,616 | FM (1 Hash): 5,295 | FM Cải tiến (128 Hash): 13,164
Đã xử lý: 300,000 dòng | Thực tế (Set): 8,694 | FM (1 Hash): 5,295 | FM Cải tiến (128 Hash): 13,734
Đã xử lý: 350,000 dòng | Thực tế (Set): 9,685 | FM (1 Hash): 5,295 | FM Cải tiến (128 Hash): 15,655
Đã xử lý: 400,000 dòng | Thực tế (Set): 10,653 | FM (1 Hash): 5,295 | FM Cải tiến (128 Hash): 16,333
Đã xử lý: 450,000 dòng | Thực tế (Set): 11,676 | FM (1 Hash): 5,295 | FM Cải tiến (128 